# Lesson 03 - Agentic Design Patterns

ਇਸ ਪਾਠ ਵਿੱਚ, ਅਸੀਂ ਪ੍ਰਭਾਵਸ਼ਾਲੀ AI ਏਜੰਟ ਬਣਾਉਣ ਲਈ ਤਿੰਨ ਬੁਨਿਆਦੀ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਨੂੰ ਅਧਿਐਨ ਕਰਦੇ ਹਾਂ:

1. **ਸਪਸ਼ਟ ਏਜੰਟ ਦਿਸ਼ਾ-ਨਿਰਦੇਸ਼** — ਏਜੰਟ ਦੇ ਵਿਹਾਰ ਨੂੰ ਮਾਰਗਦਰਸ਼ਿਤ ਕਰਨ ਵਾਲੇ ਸੁਵਿਧਾਜਨਕ, ਭੂਮਿਕਾ-ਪਰਿਭਾਸ਼ਿਤ ਪ੍ਰਾਂਪਟ ਤਿਆਰ ਕਰਨਾ  
2. **ਪਾਈਡੈਂਟਿਕ ਮਾਡਲਾਂ ਨਾਲ ਸੰਰਚਿਤ ਆਉਟਪੁੱਟ** — ਇਹ ਯਕੀਨੀ ਬਨਾਉਣਾ ਕਿ ਏਜੰਟ ਪ੍ਰਤੀਖਿਆਸ਼ੀਲ, ਪ੍ਰਮਾਣਿਤ ਡਾਟਾ ਵਾਪਸ ਕਰਦੇ ਹਨ  
3. **ਇਕਲੀ ਜਿੰਮੇਵਾਰੀ ਵਾਲੇ ਏਜੰਟ** — ਸੰਕੇਂਦ੍ਰਿਤ ਏਜੰਟ ਡਿਜ਼ਾਈਨ ਕਰਨਾ ਜੋ ਹਰ ਇੱਕ ਚੀਜ਼ ਨੂੰ ਚੰਗੀ ਤਰ੍ਹਾਂ ਕਰ ਸਕਦੇ ਹਨ  

ਅਸੀਂ ਹਰ ਪੈਟਰਨ ਨੂੰ **ਸੰਯਾਤ ਸਫਰ ਗੰਤਵ੍ਯ ਸੁਝਾਅਕਰਤਾ** ਦੇ ਦ੍ਰਿਸ਼ਟਾਂਤ ਵਿੱਚ ਲਾਗੂ ਕਰਾਂਗੇ, ਪੜ੍ਹਚੋਲਕ ਤੌਰ 'ਤੇ ਇੱਕ ਐਸਾ ਸਿਸਟਮ ਬਣਾਉਂਦੇ ਜਾਵਾਂਗੇ ਜੋ ਗੰਤਵ੍ਯ ਸਥਾਨ ਸੁਝਾਅ ਸਕਦਾ ਹੈ, ਉਪਲਬਧਤਾ ਦੀ ਜਾਂਚ ਕਰ ਸਕਦਾ ਹੈ, ਅਤੇ ਲਾਜਿਸਟਿਕਸ ਨੂੰ ਸੰਭਾਲ ਸਕਦਾ ਹੈ।


## ਸੈੱਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਪੈਟਰਨ 1: ਸਾਫ਼ ਏਜੰਟ ਨਿਰਦੇਸ਼

ਸਭ ਤੋਂ ਪ੍ਰਭਾਵਸ਼ਾਲੀ ਪੈਟਰਨ ਸਭ ਤੋਂ ਸਧਾਰਣ ਵੀ ਹੈ: ਆਪਣੇ ਏਜੰਟ ਲਈ ਸਾਫ਼, ਵਿਸਥਾਰਪੂਰਣ ਨਿਰਦੇਸ਼ ਲਿਖਣਾ।

ਚੰਗੇ ਨਿਰਦੇਸ਼ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਨ:
- **ਕੌਣ** ਏਜੰਟ ਹੈ (ਵਿਕਤੀਗਤ ਲਹਿਜ਼ਾ ਅਤੇ ਅੰਦਾਜ਼)
- **ਕੀ** ਇਹ ਕਰਨਾ ਚਾਹੀਦਾ ਹੈ (ਕਦਮ-ਦਰ-ਕਦਮ ਜ਼ਿੰਮੇਵਾਰੀਆਂ)
- **ਕਿਵੇਂ** ਇਹ ਵਰਤਾਅ ਕਰਨਾ ਚਾਹੀਦਾ ਹੈ (ਪਾਬੰਦੀਆਂ ਅਤੇ ਸਟਾਈਲ)

ਹੇਠਾਂ, ਅਸੀਂ ਇੱਕ ਯਾਤਰਾ ਕਨਸੀਅਰਜ ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜਿਸਦੀਆਂ ਵਿਆਪਕ ਨਿਰਦੇਸ਼ ਹਨ ਜੋ ਉਸਦੀ ਹਰ ਪ੍ਰਤਿਕਿਰਿਆ ਦਾ ਆਕਾਰ ਨਿਰਧਾਰਿਤ ਕਰਦੇ ਹਨ।


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Structured Output with Pydantic Models

ਫ੍ਰੀ-ਫਾਰਮ ਟੈਕਸਟ ਗੱਲਬਾਤ ਲਈ ਲਾਭਦਾਇਕ ਹੈ, ਪਰ ਡਾਊਨਸਟਰੀਮ ਸਿਸਟਮਾਂ ਨੂੰ ਸੰਰਚਿਤ ਡਾਟਾ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ।  
**Pydantic models** ਨੂੰ ਇੱਕ **ਟੂਲ ਫੰਕਸ਼ਨ** ਨਾਲ ਜੋੜ ਕੇ, ਅਸੀਂ ਕਰ ਸਕਦੇ ਹਾਂ:

- ਏਜੰਟ ਦੇ ਆਉਟਪੁੱਟ ਲਈ ਇੱਕ ਇਕਸਾਰ ਸਕੀਮਾ ਪਰਿਭਾਸ਼ਿਤ ਕਰਨਾ  
- ਜਵਾਬਾਂ ਨੂੰ ਆਪਣੇ ਆਪ ਸਹੀ ਢੰਗ ਨਾਲ ਪਰਖਣਾ  
- ਏਜੰਟ ਦੇ ਨਤੀਜੇ ਐਪਲੀਕੇਸ਼ਨ ਲੌਜਿਕ ਵਿੱਚ ਭਰੋਸੇਮੰਦ ਢੰਗ ਨਾਲ ਸ਼ਾਮਲ ਕਰਨਾ  

ਅਸੀਂ ਇੱਕ ਅਜਿਹਾ ਟੂਲ ਵੀ ਪੇਸ਼ ਕਰਦੇ ਹਾਂ ਜੋ ਮੰਜਿਲ ਦੇ ਵੇਰਵੇ ਵਾਪਸ ਕਰਦਾ ਹੈ ਤਾਂ ਜੋ ਏਜੰਟ ਆਪਣੀਆਂ ਸਿਫ਼ਾਰਸ਼ਾਂ ਨੂੰ ਅਸਲੀ ਡਾਟੇ ਵਿੱਚ ਅਧਾਰਿਤ ਕਰ ਸਕੇ।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Single Responsibility Agents

Complex tasks benefit from splitting work across multiple focused agents, each with a single responsibility:

- A **Destination Expert** that knows about places and availability
- A **Logistics Planner** that handles flights, hotels, and itineraries

This mirrors the software engineering principle of *separation of concerns* — each agent is easier to test, maintain, and improve independently.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਅਸੀਂ ਤਿੰਨ ਏਜੰਟਿਕ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨਾਂ ਨੂੰ ਯਾਤਰਾ ਸਿਫਾਰਸ਼ੀ ਦ੍ਰਿਸ਼ਟਾਂਤ 'ਚ ਲਾਗੂ ਕੀਤਾ:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | ਪ੍ਰਸੋਨਾਸਟਾ, ਜ਼ਿੰਮੇਵਾਰੀਆਂ, ਅਤੇ ਸੀਮਾਵਾਂ ਪਹਿਲਾਂ ਤੋਂ ਪਰਿਭਾਸ਼ਿਤ ਕਰੋ | ਸਥਿਰ, ਬ੍ਰਾਂਡ ਅਨੁਸਾਰ ਏਜੰਟ ਵਰਤੋਂ |
| **Structured Output** | ਜਵਾਬ ਦਾ ਫਾਰਮੈਟ ਵਜੋਂ Pydantic ਮਾਡਲਾਂ ਦੀ ਵਰਤੋਂ ਕਰੋ | ਸੱਚਾਈ ਚੈੱਕ ਕੀਤੇ, ਮਸ਼ੀਨ ਪੜ੍ਹਨ ਯੋਗ ਨਤੀਜੇ |
| **Single Responsibility** | ਹਰ ਏਜੰਟ ਨੂੰ ਇਕ ਕੇਂਦਰਿਤ ਕੰਮ ਦਿਓ | ਪ੍ਰੀਖਣ, ਦੇਖਭਾਲ ਅਤੇ ਰਚਨਾ ਲਈ ਆਸਾਨ |

ਇਹ ਪੈਟਰਨ ਕੁਦਰਤੀ ਤੌਰ 'ਤੇ ਮਿਲਦੇ ਹਨ — ਤੁਸੀਂ ਸਪਸ਼ਟ ਨਿਰਦੇਸ਼ਾਂ ਨੂੰ ਸਰਗਠਿਤ ਨਤੀਜਿਆਂ ਨਾਲ ਇਕੱਲੇ ਜਿੰਮੇਵਾਰੀ ਵਾਲੇ ਏਜੰਟ ਵਿੱਚ ਜੋੜ ਕੇ ਮਜ਼ਬੂਤ, ਉਤਪਾਦਨ-ਤਿਆਰ ਸਿਸਟਮ ਬਣਾ ਸਕਦੇ ਹੋ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
